# Lesson 2.2 - Key Vault And Kubernetes RBAC Bridge

**Level: Intermediate**

## Learning Objectives
- Provision Key Vault with globally unique naming.
- Grant Secrets Store CSI identity access to secrets.
- Configure Kubernetes provider + RBAC binding via Terraform.

**Project file(s) produced:** `terraform/foundation/keyvault.tf`, `terraform/foundation/kubeconf.tf`

## Concepts
The project enables AKS Key Vault CSI integration and then grants that managed identity read permission in Key Vault.

The Kubernetes provider uses AKS kubeconfig values so Terraform can create RBAC resources after cluster bootstrap.

In [ ]:
%%writefile terraform/foundation/keyvault.tf
data "azurerm_client_config" "current" {}

resource "random_string" "key_vault_suffix" {
  length  = 4
  special = false
  upper   = false

  keepers = {
    name_root = "kv-${var.project}-${var.landscape}-${var.region_code}"
  }
}

locals {
  key_vault_name = coalesce(
    var.key_vault_name,
    "${substr("kv-${var.project}-${var.landscape}-${var.region_code}-${var.sequence}", 0, 19)}-${random_string.key_vault_suffix.result}"
  )
}

resource "azurerm_key_vault" "this" {
  name                       = local.key_vault_name
  resource_group_name        = local.resource_group_name
  location                   = local.rg_location
  tenant_id                  = data.azurerm_client_config.current.tenant_id
  sku_name                   = var.key_vault_sku
  enable_rbac_authorization  = true
  purge_protection_enabled   = false
  soft_delete_retention_days = 7
  tags                       = var.tags
}

resource "azurerm_role_assignment" "aks_secrets_provider_kv_reader" {
  scope                = azurerm_key_vault.this.id
  role_definition_name = "Key Vault Secrets User"
  principal_id         = azurerm_kubernetes_cluster.this.key_vault_secrets_provider[0].secret_identity[0].object_id
}

In [ ]:
%%writefile terraform/foundation/kubeconf.tf
provider "kubernetes" {
  host                   = azurerm_kubernetes_cluster.this.kube_config[0].host
  client_certificate     = base64decode(azurerm_kubernetes_cluster.this.kube_config[0].client_certificate)
  client_key             = base64decode(azurerm_kubernetes_cluster.this.kube_config[0].client_key)
  cluster_ca_certificate = base64decode(azurerm_kubernetes_cluster.this.kube_config[0].cluster_ca_certificate)
}

resource "kubernetes_cluster_role_binding_v1" "flux_applier_cluster_admin" {
  metadata {
    name = "flux-applier-cluster-admin"
  }

  role_ref {
    api_group = "rbac.authorization.k8s.io"
    kind      = "ClusterRole"
    name      = "cluster-admin"
  }

  subject {
    kind      = "ServiceAccount"
    name      = "flux-applier"
    namespace = "flux-system"
  }

  depends_on = [
    azurerm_kubernetes_cluster.this,
    azurerm_kubernetes_flux_configuration.bootstrap
  ]
}

## Exercise
Create a TODO plan for adding a Key Vault secret named `hf-token` and consuming it through a SecretProviderClass in the `ai-demos` namespace.

In [ ]:
# TODO: Add az keyvault secret set command for hf-token.
# TODO: Draft SecretProviderClass YAML in a separate file.
# TODO: Mount it in a workload deployment as read-only CSI volume.

print('Write your steps, then validate with kubectl describe pod <pod-name>.')

---
**Next:** [Lesson 2.3 - Flux Bootstrap And Terraform Outputs](./lesson-2.3-flux-bootstrap-and-terraform-outputs.ipynb)